# **Analyzing Impact of Teacher Attributes on Student Evaluation**


## Regression Analysis


The goal of regression analysis is to describe the relationship between one set of variables called the dependent variables, and another set of variables, called independent or explanatory variables. When there is only one explanatory variable, it is called simple regression.


#### Objectives


* Importing Libraries
* Regression analysis in place of the t-test
* Regression analysis in place of ANOVA
* Regression analysis in place of correlation


----


#### Importing Libraries


In [5]:
import subprocess

def silent_pip_install(package_name):
    """Install a Python package using pip while suppressing all output."""
    command = ["pip", "install", package_name]
    try:
        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError:
        print(f"Failed to install '{package_name}'. Please check manually.")

silent_pip_install("numpy")
silent_pip_install("pandas")
silent_pip_install("scipy")
silent_pip_install("seaborn")
silent_pip_install("matplotlib")
silent_pip_install("statsmodels")

Importing the libraries


In [6]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

Read in the csv file from the URL


In [7]:
ratings_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/teachingratings.csv'
ratings_df = pd.read_csv(ratings_url)

### Regression with T-test: Does gender affect teaching evaluation rates?


Initially, I had used the t-test to test if there was a statistical difference in evaluations for males and females, I am now going to use regression. I stated the null hypothesis:
* $H_0: β1$ = 0 (Gender has no effect on teaching evaluation scores)
* $H_1: β1$ is not equal to 0 (Gender has an effect on teaching evaluation scores)


Female = 1 and Male = 0


In [8]:
## X is the input variables (or independent variables)
X = ratings_df['female']
## y is the target/dependent variable
y = ratings_df['eval']
## add an intercept (beta_0) to our model
X = sm.add_constant(X) 

model = sm.OLS(y, X).fit()
predictions = model.predict(X)

# Print out the statistics
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   eval   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     10.56
Date:                Mon, 04 Aug 2025   Prob (F-statistic):            0.00124
Time:                        22:08:48   Log-Likelihood:                -378.50
No. Observations:                 463   AIC:                             761.0
Df Residuals:                     461   BIC:                             769.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.0690      0.034    121.288      0.000       4.003       4.135
female        -0.1680      0.052     -3.250      0.001      -0.270      -0.066
==============================================================================
Omnibus:                       17.625   Durbin-Watson:                   1.209
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               18.970
Skew:                          -0.496   Prob(JB):                     7.60e-05
Kurtosis:                       2.981   Cond. No.                         2.47
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Conclusion:** Like the t-test, the p-value is less than the alpha (α) level = 0.05, so we reject the null hypothesis as there is evidence that there is a difference in mean evaluation scores based on gender. The coefficient -0.1680 means that female instructors get 0.168 scores less on average than male instructor.

### Regression with ANOVA: Does beauty  score for instructors  differ by age?


Stating the Hypothesis:
* $H_0: µ1 = µ2 = µ3$ (the three population means are equal)
* $H_1:$ At least one of the means differ


Grouping the data


In [9]:
ratings_df.loc[(ratings_df['age'] <= 40), 'age_group'] = '40 years and younger'
ratings_df.loc[(ratings_df['age'] > 40)&(ratings_df['age'] < 57), 'age_group'] = 'between 40 and 57 years'
ratings_df.loc[(ratings_df['age'] >= 57), 'age_group'] = '57 years and older'

Using OLS function from the statsmodel library


In [10]:
from statsmodels.formula.api import ols
lm = ols('beauty ~ age_group', data = ratings_df).fit()
table= sm.stats.anova_lm(lm)
table

,df,sum_sq,mean_sq,F,PR(>F)
age_group,2.0,20.422744,10.211372,17.597559,4.322549e-08
Residual,460.0,266.925153,0.580272,NaN,NaN


**Conclusion:** I can also see the same values for ANOVA like before and we will reject the null hypothesis since the p-value is less than 0.05 there is significant evidence that at least one of the means differ.


### Regression with ANOVA


Creating dummy variables: A dummy variable is a numeric variable that represents categorical data, such as gender, race, etc. Dummy variables are dichotomous, i.e they can take on only two quantitative values.


In [11]:
ratings_df.dtypes

minority            object
age                  int64
gender              object
credits             object
beauty             float64
eval               float64
division            object
native              object
tenure              object
students             int64
allstudents          int64
prof                 int64
PrimaryLast          int64
vismin               int64
female               int64
single_credit        int64
upper_division       int64
English_speaker      int64
tenured_prof         int64
age_group           object
dtype: object

In [12]:
X = pd.get_dummies(ratings_df[['age_group']], drop_first=True)

In [13]:
y = pd.to_numeric(ratings_df['beauty'], errors="coerce")
X = X.dropna()
y = y.dropna()
X, y = X.align(y, join='inner', axis=0)
## add an intercept (beta_0) to our model
X = sm.add_constant(X) 

model = sm.OLS(y, X.astype(float)).fit()
predictions = model.predict(X)

# Print out the statistics
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 beauty   R-squared:                       0.071
Model:                            OLS   Adj. R-squared:                  0.067
Method:                 Least Squares   F-statistic:                     17.60
Date:                Mon, 04 Aug 2025   Prob (F-statistic):           4.32e-08
Time:                        22:10:43   Log-Likelihood:                -529.47
No. Observations:                 463   AIC:                             1065.
Df Residuals:                     460   BIC:                             1077.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=====================================================================================================
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const                                 0.3362      0.072      4.692      0.000       0.195       0.477
age_group_57 years and older         -0.5820      0.099     -5.852      0.000      -0.777      -0.387
age_group_between 40 and 57 years    -0.3713      0.088     -4.237      0.000      -0.544      -0.199
==============================================================================
Omnibus:                       11.586   Durbin-Watson:                   0.434
Prob(Omnibus):                  0.003   Jarque-Bera (JB):               12.114
Skew:                           0.394   Prob(JB):                      0.00234
Kurtosis:                       2.913   Cond. No.                         4.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

We get the same results and conclusion


### Correlation: Is teaching evaluation score correlated with beauty score?


In [14]:
## X is the input variables (or independent variables)
X = ratings_df['beauty']
## y is the target/dependent variable
y = ratings_df['eval']
## add an intercept (beta_0) to our model
X = sm.add_constant(X) 

model = sm.OLS(y, X).fit()
predictions = model.predict(X)

# Print out the statistics
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   eval   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     17.08
Date:                Mon, 04 Aug 2025   Prob (F-statistic):           4.25e-05
Time:                        22:11:14   Log-Likelihood:                -375.32
No. Observations:                 463   AIC:                             754.6
Df Residuals:                     461   BIC:                             762.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.9983      0.025    157.727      0.000       3.948       4.048
beauty         0.1330      0.032      4.133      0.000       0.070       0.196
==============================================================================
Omnibus:                       15.399   Durbin-Watson:                   1.238
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               16.405
Skew:                          -0.453   Prob(JB):                     0.000274
Kurtosis:                       2.831   Cond. No.                         1.27
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Conclusion:** p < 0.05 there is evidence of correlation between beauty and evaluation scores


In [15]:
ratings_df.head(3)

,minority,age,gender,credits,beauty,eval,division,native,tenure,students,allstudents,prof,PrimaryLast,vismin,female,single_credit,upper_division,English_speaker,tenured_prof,age_group
0,yes,36,female,more,0.289916,4.3,upper,yes,yes,24,43,1,0,1,1,0,1,1,1,40 years and younger
1,yes,36,female,more,0.289916,3.7,upper,yes,yes,86,125,1,0,1,1,0,1,1,1,40 years and younger
2,yes,36,female,more,0.289916,3.6,upper,yes,yes,76,125,1,0,1,1,0,1,1,1,40 years and younger


### Does tenure affect beauty scores?
* Using α = 0.05


**Null Hypothesis**: Mean beauty scores for tenured and non-tenured instructors are equal

**Alternative Hypothesis**: There is a difference in mean beauty scores for tenured and non-tenured instructors

In [16]:
X= ratings_df["tenured_prof"]
y= ratings_df["beauty"]
X= sm.add_constant(X)

model= sm.OLS(y, X).fit()
predictions= model.predict(X)

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 beauty   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                    0.1689
Date:                Mon, 04 Aug 2025   Prob (F-statistic):              0.681
Time:                        22:11:42   Log-Likelihood:                -546.45
No. Observations:                 463   AIC:                             1097.
Df Residuals:                     461   BIC:                             1105.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0284      0.078      0.363      0.717      -0.125       0.182
tenured_prof    -0.0364      0.089     -0.411      0.681      -0.210       0.138
==============================================================================
Omnibus:                       23.184   Durbin-Watson:                   0.461
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               23.229
Skew:                           0.507   Prob(JB):                     9.03e-06
Kurtosis:                       2.583   Cond. No.                         4.05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Conclusion**: p-value is greater than 0.05, so we fail to reject the null hypothesis as there is no evidence that the mean difference of beatuty score between tenured and untenured instructors are different

### Does being an English speaker affect the number of students assigned to professors? 
 


**Null Hypothesis**: Mean number of students assigned to native English speakers vs non-native English speakers are equal

**Alternative Hypothesis**: There is a difference in mean number of students assigned to native English speakers vs non-native

In [17]:
X= ratings_df["English_speaker"]
y= ratings_df["allstudents"]
X= sm.add_constant(X)

model= sm.OLS(y, X).fit()
predictions = model.predict(X)

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            allstudents   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     3.476
Date:                Mon, 04 Aug 2025   Prob (F-statistic):             0.0629
Time:                        22:12:36   Log-Likelihood:                -2654.2
No. Observations:                 463   AIC:                             5312.
Df Residuals:                     461   BIC:                             5321.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              29.6071     14.150      2.092      0.037       1.802      57.413
English_speaker    27.2158     14.598      1.864      0.063      -1.471      55.902
==============================================================================
Omnibus:                      429.792   Durbin-Watson:                   0.708
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            10527.126
Skew:                           4.129   Prob(JB):                         0.00
Kurtosis:                      24.852   Cond. No.                         8.01
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Conclusion**

At α = 0.05, p-value is greater, we fail to reject the null hypothesis as there is no evidence that being a native English speaker or a non-native English speaker affects the number of students assigned to an instructor.
    
At α = 0.1, p-value is less, we reject the null hypothesis as there is evidence that there is a significant difference of mean number of students assigned to native English speakers vs non-native English speakers.

### What is the correlation between the number of students who participated in the evaluation survey and evaluation scores?



**Null Hypothesis**: There is a no correlation between the number of students particapted in survey and evaluation score

**Alternative Hypothesis**: There is a correlation between the number of students particapted in survey and evaluation score

In [18]:
X= ratings_df["students"]
y= ratings_df["eval"]
X= sm.add_constant(X)

model= sm.OLS(y, X).fit()
predictions= model.predict(X)

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   eval   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.5806
Date:                Mon, 04 Aug 2025   Prob (F-statistic):              0.446
Time:                        22:13:12   Log-Likelihood:                -383.46
No. Observations:                 463   AIC:                             770.9
Df Residuals:                     461   BIC:                             779.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.9823      0.033    119.689      0.000       3.917       4.048
students       0.0004      0.001      0.762      0.446      -0.001       0.002
==============================================================================
Omnibus:                       15.259   Durbin-Watson:                   1.198
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               16.283
Skew:                          -0.456   Prob(JB):                     0.000291
Kurtosis:                       2.888   Cond. No.                         74.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Conclusion**: R-square is 0.001, R will be √0.001, correlation coefficient is 0.03 (close to 0). There is a very weak correlation between the number of students who participated in the evaluation survey and evaluation scores

---
### Author: 

- [Parshv Patel](https://www.linkedin.com/in/parshv-patel-65a90326b/)
